# Qwen2.5-Coder — Natural Language → Java → C# Code Generation

Production-ready, two-stage parameter-efficient fine-tuning (PEFT/LoRA) pipeline built on
`unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit`.

| Stage | Task | Dataset | Adapter |
|-------|------|---------|---------|
| 1 | Natural Language → Java | `code_search_net` (java) | LoRA adapter on `model` |
| 2 | Java → C# | best available Java↔C# corpus (auto-fallback) | separate LoRA adapter on `model_s2` |

```
Natural Language --(Stage 1 adapter)--> Java --(Stage 2 adapter)--> C#
```

**Environment:** Designed for a single **T4 GPU** (Kaggle / Google Colab). Uses 4-bit
quantization, FP16, gradient checkpointing and 8-bit paged optimizers to fit in ~16 GB VRAM.

### Notebook layout
1. Install Dependencies · 2. Imports · 3. Configuration · 4. Logging, Reproducibility & Utilities ·
5. Dataset Loading & Validation · 6. Model Loading · 7. LoRA Configuration · 8. Prompt Generation &
Preprocessing · 9. Trainer & Training (Stage 1) · 10. Inference (Stage 1) · 11. Evaluation (Stage 1) ·
12. Save / Push (Stage 1) · 13–21. Stage 2 (Java → C#): dependencies, data, prompts, model, training,
inference, evaluation, saving, summary.

> **Reproducibility:** all RNGs are seeded from `CFG.SEED`. **Robustness:** every major stage is
> wrapped in structured logging + exception handling and degrades gracefully when an optional
> dependency or GPU feature is missing.

---

## 1. Install Dependencies

Installs Unsloth (fast 4-bit fine-tuning), the HuggingFace stack, PEFT, and code-evaluation
metrics (CodeBLEU, CodeBERTScore). **Expected runtime:** 2–4 min on a fresh Kaggle/Colab image.
Restart the kernel if prompted after the first install.

In [1]:
# Core stack: Unsloth (fast 4-bit LoRA), HF Transformers/Datasets/Accelerate/PEFT,
# bitsandbytes (4-bit + paged optimizers) and CodeBERTScore for evaluation.
!pip install unsloth transformers datasets accelerate peft bitsandbytes code-bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 24.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 30.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 30.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# CodeBLEU + tree-sitter Java grammar (pinned ABI: tree-sitter 0.22.3 / java 0.21.0).
# --force-reinstall keeps the tree-sitter ABI consistent with the C# grammar used in Stage 2.
!pip install codebleu==0.7.0 tree-sitter==0.22.3 tree-sitter-java==0.21.0 --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.2/546.2 kB 11.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0


## 2. Imports

Core libraries are imported up front; heavy/optional packages are imported lazily where used
so a missing optional metric never blocks the whole notebook.

In [4]:
# --- Standard library (modules used across multiple sections) ---
import gc
import json
import logging
import os
import random
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Optional, Sequence, Tuple

# --- Core third-party (always required) ---
import numpy as np
import torch
from datasets import Dataset, load_dataset

# NOTE: Heavy / optional dependencies (`unsloth`, `trl`, `codebleu`,
# `code_bert_score`, `sacrebleu`, `tree_sitter*`) and section-local stdlib helpers
# (`re`, `shutil`, `subprocess`, `tempfile`, `math`, `Counter`) are imported lazily
# inside the sections that use them, so a single missing optional package never
# blocks the whole notebook from loading.
print("Core imports ready | torch", torch.__version__,
      "| CUDA", torch.cuda.is_available())

Core imports ready | torch 2.10.0+cu128 | CUDA True


## 3. Configuration

Every tunable value lives in the immutable `CFG` object so experiments are reproducible and
self-documenting. Edit values here rather than scattering literals through the notebook.

In [7]:
@dataclass(frozen=True)
class Config:
    """Single source of truth for every tunable value in the notebook."""

    # --- Reproducibility ---
    SEED: int = 3407

    # --- Base model (shared by both stages) ---
    MODEL_NAME: str = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit"
    MAX_LENGTH: int = 2048

    # --- Stage 1: NL -> Java ---
    DATASET_NAME: str = "code_search_net"
    DATASET_CONFIG: str = "java"
    STAGE1_TRAIN_SAMPLES: int = 5000
    STAGE1_OUTPUT_DIR: str = "~/nl2java_checkpoints_V2"

    # --- LoRA (Stage 1) ---
    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.0
    LORA_BIAS: str = "none"
    LORA_TARGET_MODULES: Tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
    )

    # --- Training (Stage 1) ---
    EPOCHS: int = 3
    BATCH_SIZE: int = 4
    GRAD_ACCUM: int = 4
    LEARNING_RATE: float = 2e-5
    SAVE_STRATEGY: str = "epoch"
    LOGGING_STEPS: int = 20

    # --- Stage 2: Java -> C# ---
    STAGE2_MAX_SEQ_LEN: int = 1024
    STAGE2_MAX_SAMPLES: int = 6000
    STAGE2_EPOCHS: int = 2
    STAGE2_BATCH_SIZE: int = 2
    STAGE2_GRAD_ACCUM: int = 4
    STAGE2_LEARNING_RATE: float = 2e-4
    STAGE2_LORA_ALPHA: int = 16

    # --- Evaluation ---
    EVAL_SAMPLES: int = 100
    STAGE2_EVAL_SAMPLES: int = 50

    # --- Output / publishing ---
    SAVE_DIR: str = "./saved_models"
    HF_REPO: str = "shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_v2"
    # Stage 2 (Java -> C#) merged model is published to its own repo, since the two
    # stages are separate sequential adapters and cannot be fused into one model.
    HF_REPO_STAGE2: str = "shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2"


CFG = Config()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Configuration loaded. DEVICE={DEVICE} | run timestamp={TIMESTAMP}")

Configuration loaded. DEVICE=cuda | run timestamp=20260704_032934


## 4. Logging, Reproducibility & Utilities

A single structured logger replaces ad-hoc `print()` calls, all RNGs are seeded from
`CFG.SEED` for reproducibility, and small reusable helpers (`log_gpu_info`, `free_memory`)
are defined once and reused across both stages.

In [8]:
def configure_logging(name: str = "QwenCodeGen", level: int = logging.INFO) -> logging.Logger:
    """Configure a structured root logger and return a named child logger.

    Format: ``timestamp | LEVEL | name | message``. Safe to call repeatedly
    (``force=True`` resets existing handlers so re-running the cell behaves).
    """
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
        force=True,
    )
    return logging.getLogger(name)


logger = configure_logging()


def set_global_seed(seed: int) -> None:
    """Seed Python, NumPy, PyTorch (and HF) RNGs for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        from transformers import set_seed as hf_set_seed
        hf_set_seed(seed)
    except Exception:  # transformers not yet importable — non-fatal
        pass
    logger.info("Global seed set to %d", seed)


def log_gpu_info(tag: str = "") -> None:
    """Log GPU name + current memory usage, or warn if running on CPU."""
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        logger.info(
            "GPU [%s] %s | %.2f GB allocated, %.2f GB reserved, %.1f GB total",
            tag, name, alloc, reserved, total,
        )
    else:
        logger.warning(
            "CUDA not available — running on CPU (training will be very slow).")


def free_memory(*names: str, scope: dict | None = None) -> None:
    """Delete named globals, run GC and empty the CUDA cache."""
    scope = scope if scope is not None else globals()
    for n in names:
        if n in scope:
            del scope[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    log_gpu_info("after free_memory")


# Apply reproducibility settings immediately.
set_global_seed(CFG.SEED)
log_gpu_info("startup")

03:29:50 | WARNING | torchao | Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
03:29:51 | INFO | QwenCodeGen | Global seed set to 3407
03:29:51 | INFO | QwenCodeGen | GPU [startup] Tesla T4 | 0.00 GB allocated, 0.00 GB reserved, 15.6 GB total


In [11]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

## 5. Dataset Loading & Validation (Stage 1: NL → Java)

Loads the `code_search_net` Java split and validates that the required columns
(`func_documentation_string`, `func_code_string`) exist and are non-empty.
**Inputs:** none. **Outputs:** `dataset` (HF `Dataset`).

In [12]:
# Required columns for the NL -> Java task.
STAGE1_DOC_COL = "func_documentation_string"
STAGE1_CODE_COL = "func_code_string"


def load_code_dataset(split: str,
                      dataset_name: str = CFG.DATASET_NAME,
                      config: str = CFG.DATASET_CONFIG):
    """Load a split of the code dataset with logging and error handling."""
    try:
        logger.info("Loading %s/%s split='%s' ...",
                    dataset_name, config, split)
        ds = load_dataset(dataset_name, config, split=split)
        logger.info("Loaded %d examples for split '%s'.", len(ds), split)
        return ds
    except Exception:
        logger.exception("Failed to load dataset %s/%s (split=%s).",
                         dataset_name, config, split)
        raise


def validate_dataset(ds, required_columns: Sequence[str], n_check: int = 100) -> None:
    """Validate that required columns exist and have non-empty values."""
    if ds is None or len(ds) == 0:
        raise ValueError("Dataset is empty or None.")
    missing = [c for c in required_columns if c not in ds.column_names]
    if missing:
        raise ValueError(f"Dataset missing required columns: {missing} "
                         f"(available: {ds.column_names})")
    sample = ds.select(range(min(n_check, len(ds))))
    for col in required_columns:
        n_null = sum(1 for v in sample[col] if v is None or not str(v).strip())
        if n_null:
            logger.warning("Column '%s' has %d empty values in first %d rows.",
                           col, n_null, len(sample))
    logger.info("Dataset validation passed for columns: %s",
                list(required_columns))


# Load + validate the Stage 1 training split.
dataset = load_code_dataset("train")
validate_dataset(dataset, [STAGE1_DOC_COL, STAGE1_CODE_COL])

03:30:04 | INFO | QwenCodeGen | Loading code_search_net/java split='train' ...
03:30:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
03:30:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
03:30:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/code-search-net/code_search_net/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/README.md "HTTP/1.1 200 OK"
03:30:04 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/code-search-net/code_search_net/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

03:30:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 307 Temporary Redirect"
03:30:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 404 Not Found"
03:30:04 | INFO | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/code_search_net/code_search_net.py "HTTP/1.1 404 Not Found"
03:30:04 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/code_search_net/revision/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555 "HTTP/1.1 307 Temporary Redirect"
03:30:05 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/datasets/code-search-net/code_search_net/revision/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555 "HTTP/1.1 200 OK"
03:30:05 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/dat

java/train-00000-of-00001.parquet:   0%|          | 0.00/390M [00:00<?, ?B/s]

03:30:10 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/java/test-00000-of-00001.parquet "HTTP/1.1 307 Temporary Redirect"
03:30:10 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/java/test-00000-of-00001.parquet "HTTP/1.1 302 Found"


...

java/validation-00000-of-00001.parquet:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/454451 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15328 [00:00<?, ? examples/s]

03:30:15 | INFO | QwenCodeGen | Loaded 454451 examples for split 'train'.
03:30:15 | INFO | QwenCodeGen | Dataset validation passed for columns: ['func_documentation_string', 'func_code_string']


In [13]:
# Quick preview of one example (documentation -> code) to sanity-check the schema.
logger.info("Available columns: %s", dataset.column_names)
_example = dataset[2]
logger.info("Doc (truncated to instruction): %s",
            _example[STAGE1_DOC_COL].split("@param")[0].strip()[:300])
logger.info("Code (first 300 chars):\n%s", _example[STAGE1_CODE_COL][:300])

03:30:19 | INFO | QwenCodeGen | Available columns: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url']
03:30:19 | INFO | QwenCodeGen | Doc (truncated to instruction): Set Contrast adjusting factor, [-127, 127].
03:30:19 | INFO | QwenCodeGen | Code (first 300 chars):
public void setFactor(int factor) {
        this.factor = factor = Math.max(-127, Math.min(127, factor));

        if (factor > 1) {
            baseFilter.setInRed(new IntRange(factor, 255 - factor));
            baseFilter.setInGreen(new IntRange(factor, 255 - factor));
            baseFilte


## 6. Model Loading (Stage 1)

Loads the 4-bit quantized Qwen2.5-Coder base model with Unsloth. **GPU requirement:** ~3–4 GB
VRAM for the 4-bit 1.5B model. **Outputs:** `model`, `tokenizer`.

In [14]:
from unsloth import FastLanguageModel


def load_base_model(model_name: str = CFG.MODEL_NAME,
                    max_seq_length: int = CFG.MAX_LENGTH,
                    load_in_4bit: bool = True):
    """Load a 4-bit base model + tokenizer via Unsloth, with logging/validation."""
    try:
        logger.info("Loading base model '%s' (max_seq_length=%d, 4bit=%s) ...",
                    model_name, max_seq_length, load_in_4bit)
        mdl, tok = FastLanguageModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=load_in_4bit,
        )
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        logger.info("Base model loaded. EOS token id=%s", tok.eos_token_id)
        log_gpu_info("after base model load")
        return mdl, tok
    except Exception:
        logger.exception("Failed to load base model '%s'.", model_name)
        raise

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [15]:
model, tokenizer = load_base_model()

03:30:52 | INFO | QwenCodeGen | Loading base model 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit' (max_seq_length=2048, 4bit=True) ...
03:30:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:30:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
03:30:52 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
03:30:52 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


03:30:53 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
03:30:53 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 307 Temporary Redirect"
03:30:53 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 200 OK"
03:30:53 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 200 OK"
03:30:53 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/model.safetensors "HTTP/1.1 302 Found"
03:30:53 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 307 Te

model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

03:31:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/added_tokens.json "HTTP/1.1 307 Temporary Redirect"
03:31:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/added_tokens.json "HTTP/1.1 200 OK"
03:31:04 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/added_tokens.json "HTTP/1.1 200 OK"


added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

03:31:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
03:31:04 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/special_tokens_map.json "HTTP/1.1 200 OK"
03:31:04 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

03:31:05 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
03:31:05 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit "HTTP/1.1 200 OK"
03:31:05 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:31:05 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
03:31:05 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
03:31:05 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instr

## 7. LoRA Configuration (Stage 1)

Attaches a LoRA adapter (rank `CFG.LORA_R`) so only a small fraction of parameters are trained.
**Outputs:** `model` is wrapped in-place with the PEFT adapter.

In [16]:
def configure_lora(base_model,
                   r: int = CFG.LORA_R,
                   lora_alpha: int = CFG.LORA_ALPHA,
                   lora_dropout: float = CFG.LORA_DROPOUT,
                   bias: str = CFG.LORA_BIAS,
                   target_modules: Sequence[str] = CFG.LORA_TARGET_MODULES):
    """Attach a LoRA adapter to ``base_model`` and log trainable parameters."""
    try:
        logger.info("Configuring LoRA (r=%d, alpha=%d, dropout=%s) ...",
                    r, lora_alpha, lora_dropout)
        peft_model = FastLanguageModel.get_peft_model(
            model=base_model,
            r=r,
            lora_alpha=lora_alpha,
            target_modules=list(target_modules),
            lora_dropout=lora_dropout,
            bias=bias,
        )
        peft_model.print_trainable_parameters()
        logger.info("LoRA adapter attached.")
        return peft_model
    except Exception:
        logger.exception("Failed to configure LoRA adapter.")
        raise


model = configure_lora(model)

03:31:13 | INFO | QwenCodeGen | Configuring LoRA (r=16, alpha=32, dropout=0.0) ...
Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
03:31:18 | INFO | QwenCodeGen | LoRA adapter attached.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 8. Prompt Generation & Preprocessing (Stage 1)

Builds the SFT training text `### Instruction:\n\n{doc}{java_hint}\n\n### Response:\n\n{code}` using a
single shared template that is reused at **inference time**, guaranteeing the train/inference formats
match. Every instruction is anchored to Java via `STAGE1_LANG_HINT` (added in `build_stage1_prompt`)
so generic, language-less prompts don't fall back to Python. Preprocessing is **batched** for speed
and drops all columns except `prompt` in one pass. **Outputs:** `prompt_dataset` (single `prompt` column).

In [20]:
# Shared Stage 1 prompt template — used for BOTH training and inference so the
# formats can never drift apart.
STAGE1_RESPONSE_MARKER = "### Response:\n\n"

# Language hint appended to every instruction so the model is explicitly anchored to
# Java (Qwen-Coder otherwise defaults to Python for generic, language-less prompts).
# Because it is added in build_stage1_prompt(), training and inference stay identical.
STAGE1_LANG_HINT = " Write the solution in Java."


def add_java_hint(instruction: str) -> str:
    """Append the Java hint unless the instruction already mentions Java."""
    instruction = instruction.strip()
    if "java" in instruction.lower():
        return instruction
    return f"{instruction}{STAGE1_LANG_HINT}"


def build_stage1_prompt(instruction: str) -> str:
    """Prompt up to (and including) the response marker — used at inference time."""
    return f"### Instruction:\n\n{add_java_hint(instruction)}\n\n{STAGE1_RESPONSE_MARKER}"


def clean_instruction(doc: str) -> str:
    """Extract the human-readable instruction from a javadoc string."""
    return doc.split("@param")[0].strip()


def build_stage1_training_text(doc: str, code: str, eos_token: str) -> str:
    """Full prompt + target Java code + EOS — used for SFT training.
    The EOS token is appended so the model learns where the Java answer ends;
    without it the model never emits EOS at inference and keeps generating
    unrelated code until it hits max_new_tokens.
    """
    return build_stage1_prompt(clean_instruction(doc)) + code + eos_token

In [21]:
def build_prompt_dataset(ds, doc_col: str = STAGE1_DOC_COL, code_col: str = STAGE1_CODE_COL,
                         eos_token: Optional[str] = None):
    """Batched preprocessing: build the `prompt` column and drop everything else."""
    # EOS is appended so the model learns where the Java answer ends (mirrors Stage 2).
    if eos_token is None:
        eos_token = tokenizer.eos_token or "<|endoftext|>"

    def _batch(batch):
        return {"prompt": [build_stage1_training_text(d, c, eos_token)
                           for d, c in zip(batch[doc_col], batch[code_col])]}

    try:
        logger.info(
            "Building prompt dataset (batched) over %d examples ...", len(ds))
        out = ds.map(_batch, batched=True,
                     remove_columns=ds.column_names,
                     desc="Building NL->Java prompts")
        logger.info("Prompt dataset ready: %d examples, columns=%s",
                    len(out), out.column_names)
        return out
    except Exception:
        logger.exception("Failed to build prompt dataset.")
        raise


prompt_dataset = build_prompt_dataset(dataset)
logger.info("Sample prompt:\n%s", prompt_dataset[2]["prompt"][:500])

03:31:47 | INFO | QwenCodeGen | Building prompt dataset (batched) over 454451 examples ...


Building NL->Java prompts:   0%|          | 0/454451 [00:00<?, ? examples/s]

03:31:49 | INFO | QwenCodeGen | Prompt dataset ready: 454451 examples, columns=['prompt']
03:31:49 | INFO | QwenCodeGen | Sample prompt:
### Instruction:

Set Contrast adjusting factor, [-127, 127]. Write the solution in Java.

### Response:

public void setFactor(int factor) {
        this.factor = factor = Math.max(-127, Math.min(127, factor));

        if (factor > 1) {
            baseFilter.setInRed(new IntRange(factor, 255 - factor));
            baseFilter.setInGreen(new IntRange(factor, 255 - factor));
            baseFilter.setInBlue(new IntRange(factor, 255 - factor));
            baseFilter.setInGray(new IntRang


## 9. Trainer & Training (Stage 1: NL → Java)

Supervised fine-tuning with TRL's `SFTTrainer` on the first `CFG.STAGE1_TRAIN_SAMPLES` examples.
**GPU requirement:** T4 (16 GB) with 4-bit + FP16. **Expected runtime:** ~30–60 min for 3 epochs
on 5 000 samples. **Outputs:** trained `model` adapter + `stage1_trainer`.

In [22]:
import trl
from trl import SFTConfig, SFTTrainer

logger.info("Using trl version %s", trl.__version__)

03:32:11 | INFO | QwenCodeGen | Using trl version 0.24.0


In [23]:
def create_stage1_config() -> SFTConfig:
    """Build the Stage 1 SFT configuration from CFG (FP16 only when a GPU is present)."""
    return SFTConfig(
        output_dir=CFG.STAGE1_OUTPUT_DIR,
        num_train_epochs=CFG.EPOCHS,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        gradient_accumulation_steps=CFG.GRAD_ACCUM,
        learning_rate=CFG.LEARNING_RATE,
        save_strategy=CFG.SAVE_STRATEGY,
        logging_steps=CFG.LOGGING_STEPS,
        dataset_text_field="prompt",
        fp16=torch.cuda.is_available(),
        seed=CFG.SEED,
        report_to="none",
    )


config = create_stage1_config()
logger.info("Stage 1 SFTConfig ready: epochs=%d, batch=%d, grad_accum=%d, lr=%s, fp16=%s",
            config.num_train_epochs, config.per_device_train_batch_size,
            config.gradient_accumulation_steps, config.learning_rate, config.fp16)

03:32:16 | INFO | QwenCodeGen | Stage 1 SFTConfig ready: epochs=3, batch=4, grad_accum=4, lr=2e-05, fp16=True


In [24]:
def create_stage1_trainer(peft_model, tok, cfg, train_ds, n_samples: int = CFG.STAGE1_TRAIN_SAMPLES):
    """Create the Stage 1 SFTTrainer over the first ``n_samples`` examples."""
    n = min(n_samples, len(train_ds))
    if n <= 0:
        raise ValueError("No training samples available for Stage 1.")
    logger.info("Creating Stage 1 trainer over %d training samples.", n)
    common = dict(model=peft_model, args=cfg,
                  train_dataset=train_ds.select(range(n)))
    try:
        return SFTTrainer(tokenizer=tok, **common)
    except TypeError:
        # Newer TRL replaced `tokenizer` with `processing_class`.
        return SFTTrainer(processing_class=tok, **common)


stage1_trainer = create_stage1_trainer(
    model, tokenizer, config, prompt_dataset)

03:32:20 | INFO | QwenCodeGen | Creating Stage 1 trainer over 5000 training samples.
03:32:20 | INFO | unsloth.trainer | Unsloth: Padding-free batching auto-enabled for SFTTrainer instance.


Unsloth: Tokenizing ["prompt"] (num_proc=8):   0%|          | 0/5000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [25]:
def train_stage1(trainer):
    """Run Stage 1 training with logging + graceful failure."""
    try:
        logger.info("Starting Stage 1 training (NL -> Java) ...")
        log_gpu_info("before Stage 1 train")
        stats = trainer.train()
        logger.info("Stage 1 training complete. %s",
                    stats.metrics if hasattr(stats, "metrics") else stats)
        log_gpu_info("after Stage 1 train")
        return stats
    except Exception:
        logger.exception("Stage 1 training failed.")
        raise


stage1_stats = train_stage1(stage1_trainer)

03:33:29 | INFO | QwenCodeGen | Starting Stage 1 training (NL -> Java) ...
03:33:29 | INFO | QwenCodeGen | GPU [before Stage 1 train] Tesla T4 | 1.27 GB allocated, 1.29 GB reserved, 15.6 GB total
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 3 | Total steps = 939
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
20,1.864399
40,1.780516
60,1.635859
80,1.496150
100,1.362000
120,1.281531
140,1.243454
160,1.176602
180,1.190290
200,1.104617


03:46:27 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:46:27 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
03:46:27 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
03:46:27 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
Unsloth: Restored added_tokens_decoder metadata in /root/nl2java_checkpoints_V2/checkpoint-313/tokenizer_config.json.
03:46:27 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4b

## 10. Inference (Stage 1: NL → Java)

Reusable, greedy-decoding inference helpers that reuse the **same** prompt template as training.
`generate_java_from_nl` is the canonical Stage 1 generator and is reused later by the chained
NL → Java → C# pipeline. **Inputs:** natural-language string(s). **Outputs:** Java source string(s).

In [27]:
@torch.inference_mode()
def _run_generation(gen_model, gen_tokenizer, prompt_text: str, max_new_tokens: int) -> str:
    """Greedy generation primitive shared by both stages."""
    FastLanguageModel.for_inference(gen_model)
    inputs = gen_tokenizer(
        prompt_text, return_tensors="pt").to(gen_model.device)
    output = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=gen_tokenizer.eos_token_id,
        pad_token_id=gen_tokenizer.eos_token_id,
    )
    return gen_tokenizer.decode(output[0], skip_special_tokens=True)


def generate_java_from_nl(prompt: str, max_new_tokens: int = 300) -> str:
    """Stage 1: natural language -> Java. Validates input and strips the prompt echo."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a non-empty string.")
    full_prompt = build_stage1_prompt(prompt)
    try:
        decoded = _run_generation(
            model, tokenizer, full_prompt, max_new_tokens)
    except Exception:
        logger.exception(
            "NL -> Java generation failed for prompt: %r", prompt[:80])
        raise
    return decoded.split(STAGE1_RESPONSE_MARKER)[-1].strip()


def generate_java_multiple(prompts: Sequence[str], max_new_tokens: int = 300) -> List[str]:
    """Generate Java for several prompts."""
    return [generate_java_from_nl(p, max_new_tokens=max_new_tokens) for p in prompts]

In [28]:
# Demo the fine-tuned Stage 1 model on a few prompts.
_demo_prompts = [
    "write a function to check if a number is even",
    "write a function to implement binary search in Java",
    "write a function to find all permutations of a string",
]
for _p in _demo_prompts:
    logger.info("NL prompt: %s", _p)
    print(f"### {_p}\n{generate_java_from_nl(_p)}\n{'-' * 80}")

04:12:18 | INFO | QwenCodeGen | NL prompt: write a function to check if a number is even
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.mask

### write a function to check if a number is even
public static boolean isEven(int n) {
        return (n & 1) == 0;
    }
--------------------------------------------------------------------------------


04:12:22 | INFO | QwenCodeGen | NL prompt: write a function to find all permutations of a string
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### write a function to implement binary search in Java
public static <T extends Comparable<T>> int binarySearch(T[] array, T key) {
        return binarySearch(array, 0, array.length - 1, key);
    }
--------------------------------------------------------------------------------
### write a function to find all permutations of a string
public static List<String> getPermutations(String str) {
        if (str == null || str.length() == 0) {
            return new ArrayList<>();
        }
        char[] chars = str.toCharArray();
        List<String> result = new ArrayList<>();
        permute(chars, 0, chars.length - 1, result);
        return result;
    }
--------------------------------------------------------------------------------


## 11. Evaluation (Stage 1: NL → Java)

Compares the **fine-tuned** adapter against the **base** model on a held-out test subset using
**CodeBLEU** and **CodeBERTScore** (existing metrics preserved). A fresh base-model copy is loaded
for the comparison and freed afterwards. **Expected runtime:** a few minutes for
`CFG.EVAL_SAMPLES` generations on a T4.

In [30]:
# --- Stage 1 evaluation helpers (work for any model/tokenizer pair) ---
def generate_predictions(gen_model, gen_tokenizer, subset,
                         max_new_tokens: int = 300):
    """Generate Java predictions + collect references for a test subset."""
    preds: List[str] = []
    refs: List[str] = []
    for i, d in enumerate(subset):
        prompt = build_stage1_prompt(clean_instruction(d[STAGE1_DOC_COL]))
        try:
            decoded = _run_generation(
                gen_model, gen_tokenizer, prompt, max_new_tokens)
            pred = decoded.split(STAGE1_RESPONSE_MARKER)[-1].strip()
        except Exception:
            logger.exception(
                "Generation failed at index %d; using empty prediction.", i)
            pred = ""
        preds.append(pred)
        refs.append(d[STAGE1_CODE_COL])
        if (i + 1) % 25 == 0:
            logger.info("  generated %d/%d", i + 1, len(subset))
    return preds, refs


# CodeBLEU's dataflow component is unreliable when a model emits very short /
# barely-parseable output: the score is normalized by the (tiny) number of
# data-flow edges in the candidate, so a degenerate model can score spuriously
# high on dataflow while scoring ~0 on n-gram overlap. We therefore also report a
# dataflow-excluded CodeBLEU (equal weight on ngram / weighted-ngram / syntax) for
# a fairer fine-tuned-vs-base comparison. See the diagnostic cell after evaluation.
CODEBLEU_WEIGHTS_NO_DATAFLOW = (1 / 3, 1 / 3, 1 / 3, 0.0)


def compute_code_metrics(references, predictions, lang: str = "java", label: str = ""):
    """CodeBLEU + CodeBERTScore for code predictions, robust to metric failures."""
    from codebleu import calc_codebleu
    from code_bert_score import score
    report: Dict[str, object] = {}
    try:
        report["CodeBLEU"] = calc_codebleu(references, predictions, lang=lang)
        # Dataflow-excluded variant: fairer when candidates are short/degenerate.
        report["CodeBLEU_no_dataflow"] = calc_codebleu(
            references, predictions, lang=lang,
            weights=CODEBLEU_WEIGHTS_NO_DATAFLOW)
    except Exception as exc:
        logger.exception("CodeBLEU computation failed.")
        report["CodeBLEU"] = {"error": str(exc)}
        report["CodeBLEU_no_dataflow"] = {"error": str(exc)}
    try:
        # code_bert_score.score expects (candidates, references) == (predictions, references).
        _, _, f1, _ = score(predictions, references, lang=lang)
        report["CodeBERTScore_F1"] = round(float(f1.mean()), 4)
    except Exception:
        logger.exception("CodeBERTScore computation failed.")
        report["CodeBERTScore_F1"] = None
    logger.info("[%s] Stage 1 metrics: %s", label, report)
    return report


# Load a fresh base-model copy for the fine-tuned-vs-base comparison.
base_model, base_tokenizer = load_base_model()

04:17:14 | INFO | QwenCodeGen | Loading base model 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit' (max_seq_length=2048, 4bit=True) ...
04:17:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:17:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
04:17:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


04:17:14 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
04:17:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/model.safetensors "HTTP/1.1 302 Found"
04:17:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 307 Temporary Redirect"
04:17:14 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/xet-read-token/b632e7c464e861a6f1762dd396048ab1ed7a10ec "HTTP/1.1 200 OK"
04:17:14 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 200 OK"
04:17:14 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

04:17:18 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
04:17:18 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/generation_config.json "HTTP/1.1 200 OK"
04:17:18 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:17:18 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
04:17:18 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04:17:18 | INFO | httpx | HTTP Req

In [31]:
# Sanity-check the (untuned) base model on the same demo prompts.
for _p in ["write a function to check if a number is even",
           "write a function to implement binary search in Java"]:
    _prompt = build_stage1_prompt(_p)
    _decoded = _run_generation(
        base_model, base_tokenizer, _prompt, max_new_tokens=300)
    print(
        f"### [BASE] {_p}\n{_decoded.split(STAGE1_RESPONSE_MARKER)[-1].strip()}\n{'-' * 80}")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### [BASE] write a function to check if a number is even
```java
public class EvenChecker {
    /**
     * Checks if the given number is even.
     *
     * @param number The number to check.
     * @return true if the number is even, false otherwise.
     */
    public static boolean isEven(int number) {
        return number % 2 == 0;
    }

    public static void main(String[] args) {
        // Test cases to verify the correctness of the isEven function
        System.out.println(isEven(4)); // Expected: true
        System.out.println(isEven(5)); // Expected: false
        System.out.println(isEven(10)); // Expected: true
        System.out.println(isEven(-2)); // Expected: true
        System.out.println(isEven(0)); // Expected: true
    }
}
```

This code snippet defines a class `EvenChecker` with a method `isEven` that takes an integer as input and returns `true` if the number is even, and `false` otherwise. The `main` method includes test cases to demonstrate the functionality

In [32]:
# Held-out test split for evaluation.
test_data = load_code_dataset("test")
validate_dataset(test_data, [STAGE1_DOC_COL, STAGE1_CODE_COL])
test_subset = test_data.select(range(min(CFG.EVAL_SAMPLES, len(test_data))))
logger.info("Evaluating on %d test examples.", len(test_subset))

04:18:16 | INFO | QwenCodeGen | Loading code_search_net/java split='test' ...
04:18:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
04:18:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
04:18:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/code-search-net/code_search_net/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/README.md "HTTP/1.1 200 OK"
04:18:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 307 Temporary Redirect"
04:18:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/code-search-net/code_search_net/resolve/bd0cf261e357a3eb5c8fba490d23ec1a1cd59555/code_search_net.py "HTTP/1.1 404 Not Found"
04:18:16 | IN

In [33]:
# Fine-tuned model predictions on the test subset.
logger.info("Generating fine-tuned predictions ...")
predictions, references = generate_predictions(model, tokenizer, test_subset)

04:18:27 | INFO | QwenCodeGen | Generating fine-tuned predictions ...
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (h

In [34]:
# Metrics for the fine-tuned model.
finetuned_metrics = compute_code_metrics(
    references, predictions, lang="java", label="fine-tuned")
print("Fine-tuned CodeBLEU:", finetuned_metrics["CodeBLEU"])
print("Fine-tuned CodeBERTScore F1:", finetuned_metrics["CodeBERTScore_F1"])

04:33:02 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:33:02 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/config.json "HTTP/1.1 200 OK"
04:33:02 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

04:33:02 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04:33:02 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/tokenizer_config.json "HTTP/1.1 200 OK"
04:33:03 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

04:33:03 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04:33:03 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
04:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
04:33:03 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/vocab.json "HTTP/1.1 200 OK"
04:33:03 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/vocab.json "HTTP/1.1 200 OK"


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

04:33:09 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/main "HTTP/1.1 200 OK"
04:33:09 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/discussions?p=0 "HTTP/1.1 200 OK"
04:33:09 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
04:33:09 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
04:33:09 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
04:33:09 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/xet-read-token/3bb7912eb9e672e30f3b2de2fadc31e8d2cbc66b "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

04:33:14 | INFO | QwenCodeGen | [fine-tuned] Stage 1 metrics: {'CodeBLEU': {'codebleu': 0.27860605217614665, 'ngram_match_score': 0.1513896395060117, 'weighted_ngram_match_score': 0.16486227121590372, 'syntax_match_score': 0.2653180885370859, 'dataflow_match_score': 0.5328542094455853}, 'CodeBLEU_no_dataflow': {'codebleu': 0.1938566664196671, 'ngram_match_score': 0.1513896395060117, 'weighted_ngram_match_score': 0.16486227121590372, 'syntax_match_score': 0.2653180885370859, 'dataflow_match_score': 0.5328542094455853}, 'CodeBERTScore_F1': 0.7864}


Fine-tuned CodeBLEU: {'codebleu': 0.27860605217614665, 'ngram_match_score': 0.1513896395060117, 'weighted_ngram_match_score': 0.16486227121590372, 'syntax_match_score': 0.2653180885370859, 'dataflow_match_score': 0.5328542094455853}
Fine-tuned CodeBERTScore F1: 0.7864


In [35]:
# Base (untuned) model predictions on the same subset.
logger.info("Generating base-model predictions ...")
base_predictions, base_references = generate_predictions(
    base_model, base_tokenizer, test_subset)

04:33:25 | INFO | QwenCodeGen | Generating base-model predictions ...
04:38:41 | INFO | QwenCodeGen |   generated 25/100
04:43:53 | INFO | QwenCodeGen |   generated 50/100
04:49:12 | INFO | QwenCodeGen |   generated 75/100
04:54:26 | INFO | QwenCodeGen |   generated 100/100


In [36]:
# Metrics for the base model + side-by-side comparison.
base_metrics = compute_code_metrics(
    base_references, base_predictions, lang="java", label="base")
print("Base CodeBLEU:", base_metrics["CodeBLEU"])
print("Base CodeBERTScore F1:", base_metrics["CodeBERTScore_F1"])

stage1_evaluation = {"fine_tuned": finetuned_metrics, "base": base_metrics}
print("\n===== Stage 1 (NL -> Java) comparison =====")
print(f"Fine-tuned CodeBERTScore F1: {finetuned_metrics['CodeBERTScore_F1']}")
print(f"Base       CodeBERTScore F1: {base_metrics['CodeBERTScore_F1']}")

# The base model is no longer needed; free its VRAM before continuing.
free_memory("base_model", "base_tokenizer")

04:58:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04:58:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/config.json "HTTP/1.1 200 OK"
04:58:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04:58:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/neulab/codebert-java/334adda45bc5dd3593226174f7b0569ef138da91/tokenizer_config.json "HTTP/1.1 200 OK"
04:58:22 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04:58:22 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/tree/main?recursive=true&e

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

04:58:22 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/main "HTTP/1.1 200 OK"
04:58:22 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/discussions?p=0 "HTTP/1.1 200 OK"
04:58:22 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/neulab/codebert-java/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
04:58:22 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
04:58:23 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/neulab/codebert-java/resolve/refs%2Fpr%2F1/model.safetensors "HTTP/1.1 302 Found"
04:58:27 | INFO | QwenCodeGen | [base] Stage 1 metrics: {'CodeBLEU': {'codebleu': 0.25960237392846874, 'ngram_match_score': 0.0022754356737825165, 'weighted_ngram_match_score': 0.016741251292936174, 'syntax_match_score': 0.19495749047200234, 'dataflow_match_score': 0.824435318275154

Base CodeBLEU: {'codebleu': 0.25960237392846874, 'ngram_match_score': 0.0022754356737825165, 'weighted_ngram_match_score': 0.016741251292936174, 'syntax_match_score': 0.19495749047200234, 'dataflow_match_score': 0.824435318275154}
Base CodeBERTScore F1: 0.6489

===== Stage 1 (NL -> Java) comparison =====
Fine-tuned CodeBERTScore F1: 0.7864
Base       CodeBERTScore F1: 0.6489


In [37]:
# --- Fair CodeBLEU diagnostic: fine-tuned vs base -------------------------------
# CodeBLEU = equal-weighted mean of (ngram, weighted_ngram, syntax, dataflow).
# The dataflow component is unreliable when a model emits very short / degenerate
# output (denominator is the tiny number of candidate data-flow edges), so a base
# model with ~0 n-gram overlap can still score spuriously high on dataflow and thus
# "win" the aggregate CodeBLEU. Below we show the per-component breakdown and a
# dataflow-excluded CodeBLEU, which reflects the real quality gap.
def _cb(metrics):
    """Safely pull the raw CodeBLEU dict from a metrics report."""
    cb = metrics.get("CodeBLEU", {})
    return cb if isinstance(cb, dict) and "codebleu" in cb else {}


ft_cb, bs_cb = _cb(finetuned_metrics), _cb(base_metrics)
ft_nd = finetuned_metrics.get("CodeBLEU_no_dataflow", {})
bs_nd = base_metrics.get("CodeBLEU_no_dataflow", {})

print("===== Stage 1 (NL -> Java): fine-tuned vs base =====\n")
print(f"{'metric':32s}{'fine-tuned':>14s}{'base':>14s}")
print("-" * 60)
for k in ("ngram_match_score", "weighted_ngram_match_score",
          "syntax_match_score", "dataflow_match_score", "codebleu"):
    if k in ft_cb and k in bs_cb:
        print(f"{k:32s}{ft_cb[k]:>14.4f}{bs_cb[k]:>14.4f}")

print("-" * 60)
if "codebleu" in ft_nd and "codebleu" in bs_nd:
    print(f"{'codebleu (dataflow excluded)':32s}"
          f"{ft_nd['codebleu']:>14.4f}{bs_nd['codebleu']:>14.4f}")
print(f"{'CodeBERTScore F1':32s}"
      f"{(finetuned_metrics['CodeBERTScore_F1'] or float('nan')):>14.4f}"
      f"{(base_metrics['CodeBERTScore_F1'] or float('nan')):>14.4f}")

print("\nInterpretation: the fine-tuned model wins on n-gram, weighted-ngram and "
      "syntax match (and CodeBERTScore). The base model only 'wins' raw CodeBLEU "
      "via an inflated dataflow score on near-empty output. Prefer the "
      "dataflow-excluded CodeBLEU and CodeBERTScore for a fair comparison.")

===== Stage 1 (NL -> Java): fine-tuned vs base =====

metric                              fine-tuned          base
------------------------------------------------------------
ngram_match_score                       0.1514        0.0023
weighted_ngram_match_score              0.1649        0.0167
syntax_match_score                      0.2653        0.1950
dataflow_match_score                    0.5329        0.8244
codebleu                                0.2786        0.2596
------------------------------------------------------------
codebleu (dataflow excluded)            0.1939        0.0713
CodeBERTScore F1                        0.7864        0.6489

Interpretation: the fine-tuned model wins on n-gram, weighted-ngram and syntax match (and CodeBERTScore). The base model only 'wins' raw CodeBLEU via an inflated dataflow score on near-empty output. Prefer the dataflow-excluded CodeBLEU and CodeBERTScore for a fair comparison.


## 12. Save / Push Stage 1 model to the Hub

Retrieves the Hugging Face token portably (Kaggle secret → env var → Colab → prompt) and pushes a
merged FP16 export of the Stage 1 model. **Requires** a valid HF token with write access; failures
are logged and re-raised. Skip this section if you only need the local adapter (saved in Section 19).

In [38]:
# Portable Hugging Face token retrieval (works on Kaggle AND Google Colab / local).
# Order: Kaggle secret -> environment variable -> Colab userdata -> interactive prompt.
def get_hf_token() -> Optional[str]:
    """Return a HF token from the first available source, or None."""
    # 1) Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_WRITE_TOKEN")
    except Exception:
        pass
    # 2) Environment variable
    token = os.environ.get(
        "HF_WRITE_TOKEN") or os.environ.get("HF_WRITE_TOKEN")
    if token:
        return token
    # 3) Google Colab secrets
    try:
        from google.colab import userdata
        return userdata.get("HF_WRITE_TOKEN")
    except Exception:
        pass
    # 4) Interactive fallback
    from getpass import getpass
    return getpass("Enter your Hugging Face token: ")


secret_value_0 = get_hf_token()

In [40]:
# Push a merged FP16 export of the Stage 1 model to the Hub.
# Set PUSH_STAGE1_TO_HUB = False to skip (e.g. when no write token is available).
PUSH_STAGE1_TO_HUB = True

if PUSH_STAGE1_TO_HUB and secret_value_0:
    try:
        logger.info("Pushing merged Stage 1 model to '%s' ...", CFG.HF_REPO)
        model.push_to_hub_merged(
            CFG.HF_REPO,
            tokenizer,
            save_method="merged_16bit",
            token=secret_value_0,
        )
        logger.info("Push complete: %s", CFG.HF_REPO)
    except Exception:
        logger.exception("Failed to push Stage 1 model to the Hub.")
        raise
else:
    logger.warning("Skipping Hub push (PUSH_STAGE1_TO_HUB=%s, token_present=%s).",
                   PUSH_STAGE1_TO_HUB, bool(secret_value_0))

05:11:08 | INFO | QwenCodeGen | Pushing merged Stage 1 model to 'shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_v2' ...
05:11:08 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:11:08 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:11:08 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct/b9f2b422413d50e652bf90364af0efe50b48f92e/config.json "HTTP/1.1 200 OK"
05:11:08 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
05:11:08 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
05:11:

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
05:11:10 | WARNING | huggingface_hub.hf_api | No files have been modified since last commit. Skipping to prevent empty commit.
05:11:10 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_v2/revision/main "HTTP/1.1 200 OK"


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
05:11:49 | WARNING | huggingface_hub.hf_api | No files have been modified since last commit. Skipping to prevent empty commit.
05:11:49 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_v2/revision/main "HTTP/1.1 200 OK"
Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:29<00:00, 29.75s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_v2`


05:11:51 | INFO | QwenCodeGen | Push complete: shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_v2


# 13. Stage 2 — Java → C# Translation

Everything below adds a **second, independent training stage** on top of the existing
NL → Java pipeline. Nothing above is retrained or replaced.

Pipeline after this stage:

```
Natural Language  --(Stage 1 adapter)-->  Java  --(Stage 2 adapter)-->  C#
```

Design decisions:
- **Stage 1 model/adapter is left untouched.** Stage 2 uses a *separate* LoRA adapter on a
  freshly loaded copy of the same Qwen base model, so the two adapters never interfere.
- **Same base model & tokenizer** (`Qwen2.5-Coder-1.5B-Instruct-bnb-4bit`) are reused.
- **T4-friendly**: 4-bit, FP16, gradient checkpointing, `max_seq_length=CFG.STAGE2_MAX_SEQ_LEN`, 8-bit paged optimizer.
- **Dataset**: best available public Java↔C# parallel corpus, with automatic fallback.

**Sections:** 13.1 Dependencies · 14 Dataset · 15 Prompt builders · 16 Model + LoRA ·
17 Training · 18 Inference · 19 Evaluation & Saving.

In [41]:
# Stage 2 extra dependencies.
# - tree-sitter-c-sharp: pinned to 0.21.0 to match the tree-sitter 0.22.3 / tree-sitter-java 0.21.0
#   ABI already installed for Stage 1, so the C# parser used by CodeBLEU / AST metrics stays compatible.
# - sacrebleu / nltk / evaluate: BLEU + general evaluation utilities.
!pip install -q sacrebleu nltk evaluate tree-sitter-c-sharp==0.21.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.6/387.6 kB 9.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [42]:
# Free any remaining Stage 1 comparison tensors before loading a second model copy.
# (base_model/base_tokenizer were already freed at the end of Section 11; this is a safe no-op
# if they are gone, and a safeguard when Section 11 was skipped.)
free_memory("base_model", "base_tokenizer")

05:27:36 | INFO | QwenCodeGen | GPU [after free_memory] Tesla T4 | 1.42 GB allocated, 1.51 GB reserved, 15.6 GB total


## 14. Dataset Loading (Stage 2: Java ↔ C#)

Loads the best available Java↔C# parallel corpus with automatic fallback
(XLCoST → CodeTransOcean → CodeXGLUE). Each candidate is tried in turn and the first that exposes
usable `java`/`cs` columns wins. **Outputs:** `DATASET_NAME`, `raw_pairs`.

In [43]:
# ---------------------------------------------------------------------------
# Java <-> C# parallel dataset loader with automatic fallback.
#
# Preferred order (per project spec): XLCoST -> CodeTransOcean -> AVATAR.
# NOTE: AVATAR is a Java<->Python corpus and has no C#, so for the Java->C#
#       task the reliable anchor is CodeXGLUE "code-to-code-trans", which is the
#       canonical Java<->C# translation benchmark and is always available on the Hub.
#
# Each candidate is tried in turn; the first one that loads AND exposes usable
# java/csharp columns wins. Everything is wrapped in try/except so an unavailable
# dataset simply falls through to the next.
# ---------------------------------------------------------------------------

# Each candidate: (label, kwargs for load_dataset, java_column, csharp_column)
DATASET_CANDIDATES = [
    # 1) XLCoST (preferred) — community parallel Java/C# mirrors.
    ("XLCoST", dict(path="codeparrot/xlcost-text-to-code",
     name="Csharp-program-level"), None, None),
    # 2) CodeTransOcean — multilingual code translation.
    ("CodeTransOcean", dict(path="WeixiangYan/CodeTransOcean",
     name="MultilingualTrans"), "java", "csharp"),
    # 3) CodeXGLUE Java<->C# (guaranteed anchor).
    ("CodeXGLUE-Java-CS", dict(path="google/code_x_glue_cc_code_to_code_trans"), "java", "cs"),
]


def _normalize_split(ds, java_col: Optional[str], csharp_col: Optional[str]):
    """Return a list of {'java', 'cs'} dicts, or None if columns are not usable."""
    cols = set(ds.column_names)
    # Auto-detect column names if not provided.
    if java_col is None or java_col not in cols:
        java_col = next((c for c in cols if c.lower() in (
            "java", "java_code", "src", "source")), None)
    if csharp_col is None or csharp_col not in cols:
        csharp_col = next((c for c in cols if c.lower() in (
            "cs", "csharp", "c#", "cs_code", "tgt", "target")), None)
    if not java_col or not csharp_col:
        return None
    pairs = [{"java": j, "cs": c}
             for j, c in zip(ds[java_col], ds[csharp_col])
             if isinstance(j, str) and isinstance(c, str) and j.strip() and c.strip()]
    return pairs or None


def load_java_csharp_dataset():
    """Try each candidate dataset in order; return (label, {split: pairs})."""
    for label, kwargs, java_col, csharp_col in DATASET_CANDIDATES:
        try:
            logger.info("Trying Java/C# dataset: %s (%s) ...",
                        label, kwargs.get("path"))
            raw = load_dataset(**kwargs)
            split_names = list(raw.keys()) if hasattr(
                raw, "keys") else ["train"]
            collected = {}
            for split in split_names:
                pairs = _normalize_split(raw[split], java_col, csharp_col)
                if pairs:
                    collected[split] = pairs
            if collected:
                total = sum(len(v) for v in collected.values())
                logger.info("Loaded %s: %d Java/C# pairs across splits %s",
                            label, total, list(collected))
                return label, collected
            logger.warning(
                "%s has no usable Java/C# columns, skipping.", label)
        except Exception as exc:
            logger.warning("%s unavailable (%s: %s). Falling back.",
                           label, type(exc).__name__, exc)
    raise RuntimeError(
        "No Java<->C# dataset could be loaded from any candidate.")


DATASET_NAME, raw_pairs = load_java_csharp_dataset()
logger.info("Selected Stage 2 dataset: %s", DATASET_NAME)

05:28:38 | INFO | QwenCodeGen | Trying Java/C# dataset: XLCoST (codeparrot/xlcost-text-to-code) ...
05:28:38 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/codeparrot/xlcost-text-to-code/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
05:28:38 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/codeparrot/xlcost-text-to-code/60c5c133f043a5cffe162f9de1c62b9d88f309cf/README.md "HTTP/1.1 200 OK"
05:28:39 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/codeparrot/xlcost-text-to-code/60c5c133f043a5cffe162f9de1c62b9d88f309cf/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

05:28:39 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/codeparrot/xlcost-text-to-code/resolve/60c5c133f043a5cffe162f9de1c62b9d88f309cf/xlcost-text-to-code.py "HTTP/1.1 307 Temporary Redirect"
05:28:39 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/codeparrot/xlcost-text-to-code/60c5c133f043a5cffe162f9de1c62b9d88f309cf/xlcost-text-to-code.py "HTTP/1.1 200 OK"
05:28:39 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/codeparrot/xlcost-text-to-code/60c5c133f043a5cffe162f9de1c62b9d88f309cf/xlcost-text-to-code.py "HTTP/1.1 200 OK"


xlcost-text-to-code.py: 0.00B [00:00, ?B/s]

05:28:39 | WARNING | QwenCodeGen | XLCoST unavailable (RuntimeError: Dataset scripts are no longer supported, but found xlcost-text-to-code.py). Falling back.
05:28:39 | INFO | QwenCodeGen | Trying Java/C# dataset: CodeTransOcean (WeixiangYan/CodeTransOcean) ...
05:28:39 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/datasets/WeixiangYan/CodeTransOcean/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
05:28:39 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/WeixiangYan/CodeTransOcean/8985d0851100f08e94735b2f2556a795ba8c69bb/README.md "HTTP/1.1 200 OK"
05:28:39 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/WeixiangYan/CodeTransOcean/8985d0851100f08e94735b2f2556a795ba8c69bb/README.md "HTTP/1.1 200 OK"


...

data/test-00000-of-00001.parquet:   0%|          | 0.00/170k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10300 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

05:28:42 | INFO | QwenCodeGen | Loaded CodeXGLUE-Java-CS: 11800 Java/C# pairs across splits ['train', 'validation', 'test']
05:28:42 | INFO | QwenCodeGen | Selected Stage 2 dataset: CodeXGLUE-Java-CS


## 15. Prompt Builders & Preprocessing (Stage 2)

Shared Java→C# prompt builders (used for **both** training and inference) followed by cleaning,
de-duplication, filtering and a deterministic train/validation split. **Outputs:** `train_pairs`,
`val_pairs` and the Stage 2 prompt helpers.

In [44]:
# ---------------------------------------------------------------------------
# Shared Stage 2 prompt builders (used for BOTH training and inference so the
# format is guaranteed to match — a common cause of degraded generations).
# ---------------------------------------------------------------------------
STAGE2_INSTRUCTION = "Translate the following Java code into equivalent C#."
STAGE2_RESPONSE_MARKER = "### Response\n"

# Language hint appended to the instruction so the model is explicitly anchored to
# C# (mirrors Stage 1's Java hint). Added in build_stage2_inference_prompt() so the
# training and inference formats can never drift apart.
STAGE2_LANG_HINT = " Write the solution in C#."


def add_csharp_hint(instruction: str) -> str:
    """Append the C# hint unless the instruction already mentions C#."""
    instruction = instruction.strip()
    lowered = instruction.lower()
    if "c#" in lowered or "csharp" in lowered:
        return instruction
    return f"{instruction}{STAGE2_LANG_HINT}"


def build_stage2_inference_prompt(java_code: str) -> str:
    """Prompt up to (and including) the response marker — used at inference time."""
    return (
        "### Instruction\n"
        f"{add_csharp_hint(STAGE2_INSTRUCTION)}\n\n"
        "### Java\n"
        f"{java_code.strip()}\n\n"
        "### Response\n"
    )


def build_stage2_training_text(java_code: str, csharp_code: str, eos_token: str) -> str:
    """Full prompt + target + EOS — used for SFT training."""
    return build_stage2_inference_prompt(java_code) + csharp_code.strip() + eos_token

In [46]:
# ---------------------------------------------------------------------------
# Clean + de-duplicate + filter + shuffle + split.
# ---------------------------------------------------------------------------
random.seed(CFG.SEED)

# Flatten all collected splits into one pool, then build our own train/val split
# so the procedure is identical regardless of which dataset was selected.
all_pairs = []
for split_pairs in raw_pairs.values():
    all_pairs.extend(split_pairs)

logger.info("Raw Java/C# pairs: %d", len(all_pairs))


def is_valid_pair(java_code: str, csharp_code: str) -> bool:
    """Cheap quality filter: non-empty, not trivially short, and 'looks like code'."""
    j, c = java_code.strip(), csharp_code.strip()
    if not j or not c:                       # remove empties
        return False
    if len(j) < 10 or len(c) < 10:           # remove trivially short / junk
        return False
    # cheap "looks like code" check — must contain a brace or semicolon
    if not any(tok in j for tok in (";", "{", "}")):
        return False
    if not any(tok in c for tok in (";", "{", "}")):
        return False
    return True


# Filter invalid, then de-duplicate on (java, cs).
seen = set()
clean_pairs = []
for p in all_pairs:
    if not is_valid_pair(p["java"], p["cs"]):
        continue
    key = (p["java"].strip(), p["cs"].strip())
    if key in seen:
        continue
    seen.add(key)
    clean_pairs.append({"java": p["java"].strip(), "cs": p["cs"].strip()})

random.shuffle(clean_pairs)
logger.info("Clean, de-duplicated pairs: %d", len(clean_pairs))

# Cap the training pool to keep T4 training fast. Configured via CFG.STAGE2_MAX_SAMPLES.
clean_pairs = clean_pairs[:CFG.STAGE2_MAX_SAMPLES]

# Guard: need a minimum number of pairs to train at all.
if len(clean_pairs) < 20:
    raise ValueError(
        f"Too few valid Java/C# pairs ({len(clean_pairs)}) to train Stage 2. "
        "Check the dataset loader / cleaning filters.")

# 95/5 train/validation split, but never let validation exceed 20% of the pool
# (otherwise a small fallback dataset could leave train_pairs empty).
val_size = min(max(50, int(0.05 * len(clean_pairs))),
               max(1, len(clean_pairs) // 5))
val_pairs = clean_pairs[:val_size]
train_pairs = clean_pairs[val_size:]
if not train_pairs:
    raise ValueError("Training split is empty after the validation cut.")
logger.info("Stage 2 split -> Train: %d | Validation: %d",
            len(train_pairs), len(val_pairs))

05:29:06 | INFO | QwenCodeGen | Raw Java/C# pairs: 11800
05:29:06 | INFO | QwenCodeGen | Clean, de-duplicated pairs: 11800
05:29:06 | INFO | QwenCodeGen | Stage 2 split -> Train: 5700 | Validation: 300


## 16. Model Loading + LoRA (Stage 2)

Loads a **fresh** 4-bit copy of the same Qwen base model and attaches a **separate** LoRA adapter,
guaranteeing the Stage 1 adapter is never modified. **Outputs:** `model_s2`, `tokenizer_s2`,
`train_dataset_s2`, `val_dataset_s2`.

In [48]:
# ---------------------------------------------------------------------------
# Load a FRESH copy of the same Qwen base model for Stage 2 and attach a SEPARATE
# LoRA adapter. This guarantees the Stage 1 adapter (model / tokenizer) is never
# modified. The tokenizer is identical to Stage 1's since the base model is the same.
# ---------------------------------------------------------------------------
STAGE2_MAX_SEQ_LEN = CFG.STAGE2_MAX_SEQ_LEN

try:
    model_s2, tokenizer_s2 = FastLanguageModel.from_pretrained(
        model_name=CFG.MODEL_NAME,
        max_seq_length=STAGE2_MAX_SEQ_LEN,
        load_in_4bit=True,
    )
    if tokenizer_s2.pad_token is None:
        tokenizer_s2.pad_token = tokenizer_s2.eos_token

    # Separate LoRA adapter for Stage 2 (recommended config).
    model_s2 = FastLanguageModel.get_peft_model(
        model_s2,
        r=CFG.LORA_R,
        lora_alpha=CFG.STAGE2_LORA_ALPHA,
        lora_dropout=CFG.LORA_DROPOUT,
        bias=CFG.LORA_BIAS,
        target_modules=list(CFG.LORA_TARGET_MODULES),
        # memory-efficient gradient checkpointing
        use_gradient_checkpointing="unsloth",
        max_seq_length=STAGE2_MAX_SEQ_LEN,
        random_state=CFG.SEED,
    )
    model_s2.print_trainable_parameters()
    logger.info("Stage 2 model + LoRA ready (separate adapter).")
    log_gpu_info("after Stage 2 model load")
except Exception:
    logger.exception("Failed to load Stage 2 model / attach LoRA adapter.")
    raise

05:31:36 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:31:36 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
05:31:36 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


05:31:37 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
05:31:37 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/model.safetensors "HTTP/1.1 302 Found"
05:31:37 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "HTTP/1.1 307 Temporary Redirect"
05:31:37 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/xet-read-token/b632e7c464e861a6f1762dd396048ab1ed7a10ec "HTTP/1.1 200 OK"
05:31:37 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/config.json "HTTP/1.1 307 Temporary Redirect"
05:31:37 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unslothai/kaggle/b632e7c464e861a6f1762dd396048ab1ed7a10ec/.gitattributes "H

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

05:31:41 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
05:31:41 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/generation_config.json "HTTP/1.1 200 OK"
05:31:41 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:31:41 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
05:31:41 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05:31:41 | INFO | httpx | HTTP Req

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [50]:
# Build HuggingFace Datasets with a single "text" field for SFT.
# EOS is appended so the model learns where the C# answer ends.
_eos = tokenizer_s2.eos_token or "<|endoftext|>"


def _to_text_dataset(pairs):
    texts = [build_stage2_training_text(
        p["java"], p["cs"], _eos) for p in pairs]
    return Dataset.from_dict({"text": texts})


train_dataset_s2 = _to_text_dataset(train_pairs)
val_dataset_s2 = _to_text_dataset(val_pairs)

logger.info("Stage 2 text datasets built: train=%d, val=%d",
            len(train_dataset_s2), len(val_dataset_s2))
logger.info("Example Stage 2 training sample:\n%s",
            train_dataset_s2[0]["text"][:800])

05:32:03 | INFO | QwenCodeGen | Stage 2 text datasets built: train=5700, val=300
05:32:03 | INFO | QwenCodeGen | Example Stage 2 training sample:
### Instruction
Translate the following Java code into equivalent C#.

### Java
public OldFormulaRecord(RecordInputStream ris) {super(ris, ris.getSid() == biff2_sid);if (isBiff2()) {field_4_value = ris.readDouble();} else {long valueLongBits  = ris.readLong();specialCachedValue = FormulaSpecialCachedValue.create(valueLongBits);if (specialCachedValue == null) {field_4_value = Double.longBitsToDouble(valueLongBits);}}if (isBiff2()) {field_5_options = (short)ris.readUByte();} else {field_5_options = ris.readShort();}int expression_len = ris.readShort();int nBytesAvailable = ris.available();field_6_parsed_expr = Formula.read(expression_len, ris, nBytesAvailable);}

### Response
public OldFormulaRecord(RecordInputStream ris) :base(ris, ris.Sid == biff2_sid){;if (IsBiff2){field_4_value = ris.Rea


## 17. Trainer & Training (Stage 2: Java → C#)

Version-tolerant `SFTConfig`/`SFTTrainer` construction (handles the `eval_strategy` rename and the
`tokenizer`→`processing_class` rename) tuned for a single T4. **Expected runtime:** ~20–40 min for
2 epochs. **Outputs:** trained `model_s2` adapter + `stage2_stats`.

In [51]:
# ---------------------------------------------------------------------------
# Stage 2 SFT configuration - tuned for a single T4 (Colab / Kaggle).
#
# SFTConfig / TrainingArguments keyword names drift across trl & transformers
# versions (e.g. eval_strategy vs evaluation_strategy, and some keys exist only
# in newer releases). build_sft_config() below filters to the keys the installed
# SFTConfig actually accepts and handles the eval_strategy rename, so this cell
# will not raise a TypeError on older/newer stacks.
# ---------------------------------------------------------------------------


def pick_optimizer() -> str:
    """paged_adamw_8bit when bitsandbytes is available, else adamw_torch."""
    try:
        import bitsandbytes  # noqa: F401
        return "paged_adamw_8bit"
    except Exception:
        return "adamw_torch"


def _accepted_config_keys():
    keys = set(getattr(SFTConfig, "__dataclass_fields__", {}).keys())
    try:
        from transformers import TrainingArguments
        keys |= set(getattr(TrainingArguments,
                    "__dataclass_fields__", {}).keys())
    except Exception:
        pass
    return keys


def build_sft_config(**kwargs) -> "SFTConfig":
    """Construct an SFTConfig, tolerating version-specific keyword differences."""
    keys = _accepted_config_keys()
    # Handle the evaluation_strategy -> eval_strategy rename in both directions.
    if "eval_strategy" in kwargs and "eval_strategy" not in keys and "evaluation_strategy" in keys:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    # Drop any keyword the installed SFTConfig does not understand.
    if keys:
        dropped = [k for k in kwargs if k not in keys]
        for k in dropped:
            kwargs.pop(k)
        if dropped:
            logger.warning(
                "Dropped unsupported SFTConfig args for this version: %s", dropped)
    return SFTConfig(**kwargs)


def build_sft_trainer(cfg):
    """Create the SFTTrainer, tolerating the tokenizer -> processing_class rename."""
    common = dict(
        model=model_s2,
        args=cfg,
        train_dataset=train_dataset_s2,
        eval_dataset=val_dataset_s2,
    )
    try:
        return SFTTrainer(tokenizer=tokenizer_s2, **common)
    except TypeError:
        # Newer TRL removed the `tokenizer` argument in favour of processing_class.
        return SFTTrainer(processing_class=tokenizer_s2, **common)


use_fp16 = torch.cuda.is_available()

stage2_config = build_sft_config(
    output_dir="./java2csharp_checkpoints",
    num_train_epochs=CFG.STAGE2_EPOCHS,
    per_device_train_batch_size=CFG.STAGE2_BATCH_SIZE,
    gradient_accumulation_steps=CFG.STAGE2_GRAD_ACCUM,   # effective batch size 8
    learning_rate=CFG.STAGE2_LEARNING_RATE,              # standard LoRA LR
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=CFG.LOGGING_STEPS,
    save_strategy="epoch",
    eval_strategy="epoch",
    optim=pick_optimizer(),
    fp16=use_fp16,                          # FP16 on CUDA; never bf16/TPU
    bf16=False,
    gradient_checkpointing=True,
    max_seq_length=STAGE2_MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    report_to="none",
    seed=CFG.SEED,
)

stage2_trainer = build_sft_trainer(stage2_config)
logger.info("Stage 2 trainer ready. Optimizer=%s | fp16=%s",
            stage2_config.optim, stage2_config.fp16)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/5700 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/300 [00:00<?, ? examples/s]

05:32:41 | INFO | QwenCodeGen | Stage 2 trainer ready. Optimizer=OptimizerNames.PAGED_ADAMW_8BIT | fp16=True


In [52]:
# Train the Stage 2 (Java -> C#) adapter.
try:
    logger.info("Starting Stage 2 training (Java -> C#) ...")
    log_gpu_info("before Stage 2 train")
    stage2_stats = stage2_trainer.train()
    logger.info("Stage 2 training complete. %s",
                stage2_stats.metrics if hasattr(stage2_stats, "metrics") else stage2_stats)
    log_gpu_info("after Stage 2 train")
except Exception:
    logger.exception("Stage 2 training failed.")
    raise

05:32:48 | INFO | QwenCodeGen | Starting Stage 2 training (Java -> C#) ...
05:32:48 | INFO | QwenCodeGen | GPU [before Stage 2 train] Tesla T4 | 2.67 GB allocated, 2.77 GB reserved, 15.6 GB total


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!
{'loss': '1.69', 'grad_norm': '1.452', 'learning_rate': '8.837e-05', 'epoch': '0.02807'}
{'loss': '0.7633', 'grad_norm': '0.5884', 'learning_rate': '0.0001814', 'epoch': '0.05614'}
{'loss': '0.6304', 'grad_norm': '0.408', 'learning_rate': '0.0001999', 'epoch': '0.08421'}
{'loss': '0.5747', 'grad_norm': '0.3353', 'learning_rate': '0.0001997', 'epoch': '0.1123'}
{'loss': '0.5658', 'grad_norm': '0.4089', 'learning_rate': '0.0001992', 'epoch': '0.1404'}
{'loss': '0.5576', 'grad_norm': '0.3633', 'learning_rate': '0.0001985', 'epoch': '0.1684'}
{'loss': '0.5463', 'grad_norm': '0.4219', 'learning_rate': '0.0001976', 'epoch': '0.1965'}
{'loss': '0.5724', 'grad_norm': '0.6796', 'learning_rate': '0.0001965', 'epoch': '0.2246'}
{'loss': '0.5723', 'grad_norm': '0.3258', 'learning_rate': '0.0001953', 'epoch': '0.2526'}
{'loss': '0.5093', 'grad_norm': '0.5862', 'learnin

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

{'eval_loss': '0.44', 'eval_runtime': '17.18', 'eval_samples_per_second': '17.46', 'eval_steps_per_second': '4.364', 'epoch': '1'}


05:55:47 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:55:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
05:55:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05:55:48 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"


...

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

{'eval_loss': '0.4182', 'eval_runtime': '17.28', 'eval_samples_per_second': '17.36', 'eval_steps_per_second': '4.341', 'epoch': '2'}


06:20:51 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06:20:51 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"


{'train_runtime': '2882', 'train_samples_per_second': '3.956', 'train_steps_per_second': '0.495', 'train_loss': '0.4504', 'epoch': '2'}


06:20:52 | INFO | QwenCodeGen | Stage 2 training complete. {'train_runtime': 2881.6613, 'train_samples_per_second': 3.956, 'train_steps_per_second': 0.495, 'train_loss': 0.45035252831928524, 'epoch': 2.0}
06:20:52 | INFO | QwenCodeGen | GPU [after Stage 2 train] Tesla T4 | 2.69 GB allocated, 2.82 GB reserved, 15.6 GB total


## 18. Reusable inference — chained NL → Java → C#

Three reusable functions. The user only ever needs to supply natural language;
`generate_csharp_from_nl` internally chains Stage 1 and Stage 2. The Stage 1 generator and the
shared `_run_generation` primitive are reused from Section 10 (no duplication).

In [53]:
# ---------------------------------------------------------------------------
# Reusable inference helpers (Stage 2 + chained pipeline).
#   generate_java_from_nl(prompt)      -> Stage 1 (defined in Section 10)
#   generate_csharp_from_java(java)    -> Stage 2 (Java -> C#)
#   generate_csharp_from_nl(prompt)    -> chains Stage 1 then Stage 2
# `_run_generation` and `generate_java_from_nl` are reused from Section 10.
# ---------------------------------------------------------------------------
def generate_csharp_from_java(java_code: str, max_new_tokens: int = 400) -> str:
    """Stage 2: Java -> C#."""
    if not isinstance(java_code, str) or not java_code.strip():
        raise ValueError("java_code must be a non-empty string.")
    full_prompt = build_stage2_inference_prompt(java_code)
    try:
        decoded = _run_generation(
            model_s2, tokenizer_s2, full_prompt, max_new_tokens)
    except Exception:
        logger.exception("Java -> C# generation failed.")
        raise
    return decoded.split(STAGE2_RESPONSE_MARKER)[-1].strip()


def generate_csharp_from_nl(prompt: str,
                            java_max_new_tokens: int = 300,
                            csharp_max_new_tokens: int = 400) -> Dict[str, str]:
    """Full chain: NL -> Java -> C#. Returns both intermediate Java and final C#."""
    java_code = generate_java_from_nl(
        prompt, max_new_tokens=java_max_new_tokens)
    csharp_code = generate_csharp_from_java(
        java_code, max_new_tokens=csharp_max_new_tokens)
    return {"natural_language": prompt, "java": java_code, "csharp": csharp_code}

In [54]:
print(train_dataset_s2[0]["text"])

### Instruction
Translate the following Java code into equivalent C#.

### Java
public OldFormulaRecord(RecordInputStream ris) {super(ris, ris.getSid() == biff2_sid);if (isBiff2()) {field_4_value = ris.readDouble();} else {long valueLongBits  = ris.readLong();specialCachedValue = FormulaSpecialCachedValue.create(valueLongBits);if (specialCachedValue == null) {field_4_value = Double.longBitsToDouble(valueLongBits);}}if (isBiff2()) {field_5_options = (short)ris.readUByte();} else {field_5_options = ris.readShort();}int expression_len = ris.readShort();int nBytesAvailable = ris.available();field_6_parsed_expr = Formula.read(expression_len, ris, nBytesAvailable);}

### Response
public OldFormulaRecord(RecordInputStream ris) :base(ris, ris.Sid == biff2_sid){;if (IsBiff2){field_4_value = ris.ReadDouble();}else{long valueLongBits = ris.ReadLong();specialCachedValue = SpecialCachedValue.Create(valueLongBits);if (specialCachedValue == null){field_4_value = BitConverter.Int64BitsToDouble(valueLo

In [55]:
# Quick sanity check of each mode.

# Mode 1: NL -> Java only
print("=== Mode 1: NL -> Java ===")
print(generate_java_from_nl("write a function to check if a number is prime"))

# Stage 2 only: Java -> C#
print("\n=== Stage 2: Java -> C# ===")
sample_java = """public int add(int a, int b) {
    return a + b;
}"""
print(generate_csharp_from_java(sample_java))

# Mode 2: NL -> Java -> C# (chained)
print("\n=== Mode 2: NL -> Java -> C# (chained) ===")
result = generate_csharp_from_nl(
    "write a function to compute the factorial of a number")
print("\n--- Java ---\n", result["java"])
print("\n--- C# ---\n", result["csharp"])

=== Mode 1: NL -> Java ===


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


public static boolean isPrime(int n) {
        if (n < 2)
            return false;
        for (int i = 2; i <= Math.sqrt(n); i++)
            if (n % i == 0)
                return false;
        return true;
    }

=== Stage 2: Java -> C# ===
public virtual int Add(int a, int b){return a + b;}

=== Mode 2: NL -> Java -> C# (chained) ===

--- Java ---
 public static long factorial(int n) {
        if (n < 0)
            throw new IllegalArgumentException("Factorial is not defined for negative numbers");
        else if (n == 0 || n == 1)
            return 1;
        else
            return n * factorial(n - 1);
    }

--- C# ---
 public static long Factorial(int n){if (n < 0){throw new ArgumentException("Factorial is not defined for negative numbers");}else{if (n == 0 || n == 1){return 1;}else{return n * Factorial(n - 1);}}}


## 19. Stage 2 evaluation

Metrics computed on a held-out validation subset:
BLEU, CodeBLEU, Exact Match, Syntax Accuracy, AST Similarity, and Compilation Rate.

> Compilation Rate uses a real C# compiler (`dotnet` / `csc` / `mcs`) when one is present.
> On vanilla Kaggle/Colab images no C# toolchain is installed, so it transparently falls
> back to a tree-sitter **syntax-validity proxy** (clearly labelled in the report).

In [58]:
# ---------------------------------------------------------------------------
# Evaluation metric helpers for C#.
# ---------------------------------------------------------------------------
import math
import re
import shutil
import subprocess
import tempfile
from collections import Counter

import sacrebleu
from codebleu import calc_codebleu


def _normalize_code(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())


def get_csharp_parser():
    """Best-effort tree-sitter C# parser; returns None if unavailable."""
    try:
        from tree_sitter import Language, Parser
        import tree_sitter_c_sharp as tscs
        lang = Language(tscs.language())
        try:
            return Parser(lang)              # tree-sitter >= 0.22 API
        except TypeError:
            p = Parser()
            p.set_language(lang)             # older API
            return p
    except Exception as exc:
        print("C# tree-sitter parser unavailable:", exc)
        return None


CS_PARSER = get_csharp_parser()


def _collect_node_types(node, acc):
    acc.append(node.type)
    for child in node.children:
        _collect_node_types(child, acc)


def _ast_node_types(code: str):
    if CS_PARSER is None:
        return None
    try:
        tree = CS_PARSER.parse(bytes(code, "utf8"))
        acc = []
        _collect_node_types(tree.root_node, acc)
        return acc
    except Exception:
        return None


def _parse_is_valid(code: str):
    if CS_PARSER is None:
        return None
    try:
        tree = CS_PARSER.parse(bytes(code, "utf8"))
        return not tree.root_node.has_error
    except Exception:
        return False


def _bag_cosine(a, b):
    ca, cb = Counter(a), Counter(b)
    keys = set(ca) | set(cb)
    dot = sum(ca[k] * cb[k] for k in keys)
    na = math.sqrt(sum(v * v for v in ca.values()))
    nb = math.sqrt(sum(v * v for v in cb.values()))
    return dot / (na * nb) if na and nb else 0.0


# ---- Compilation rate (real compiler if available, else syntax proxy) ----
def _find_csharp_compiler():
    for tool in ("dotnet", "csc", "mcs"):
        if shutil.which(tool):
            return tool
    return None


CSHARP_COMPILER = _find_csharp_compiler()


def _wrap_csharp(code: str) -> str:
    """Wrap a bare snippet/method into a compilable class skeleton."""
    if re.search(r"\b(class|struct|interface|enum|namespace)\b", code):
        return code
    return ("using System;\nusing System.Collections.Generic;\n"
            "public class Wrapper {\n" + code + "\n}\n")


def _compiles_with_mono(code: str) -> bool:
    with tempfile.TemporaryDirectory() as tmp:
        src = f"{tmp}/Program.cs"
        with open(src, "w", encoding="utf-8") as fh:
            fh.write(_wrap_csharp(code))
        try:
            out = subprocess.run([CSHARP_COMPILER, "-target:library", src],
                                 capture_output=True, timeout=30)
            return out.returncode == 0
        except Exception:
            return False


def compilation_rate(preds):
    """Returns (rate, method_label)."""
    if CSHARP_COMPILER in ("csc", "mcs"):
        oks = [_compiles_with_mono(p) for p in preds]
        return sum(oks) / len(preds), f"real-compiler({CSHARP_COMPILER})"
    # No usable standalone compiler -> tree-sitter syntax proxy.
    valids = [v for v in (_parse_is_valid(p) for p in preds) if v is not None]
    if not valids:
        return None, "unavailable"
    return sum(valids) / len(valids), "syntax-proxy(tree-sitter)"


def evaluate_csharp(predictions, references):
    report = {}

    # BLEU (sacrebleu corpus BLEU).
    report["BLEU"] = round(sacrebleu.corpus_bleu(
        predictions, [references]).score, 4)

    # CodeBLEU for C#.
    try:
        cb = calc_codebleu(references, predictions, lang="c_sharp")
        report["CodeBLEU"] = {k: round(v, 4) for k, v in cb.items()}
    except Exception as exc:
        report["CodeBLEU"] = {"error": str(exc)}

    # Exact match (whitespace-normalized).
    report["ExactMatch"] = round(
        sum(_normalize_code(p) == _normalize_code(r)
            for p, r in zip(predictions, references)) / len(predictions), 4)

    # Syntax accuracy (fraction that parse without errors).
    syn = [v for v in (_parse_is_valid(p)
                       for p in predictions) if v is not None]
    report["SyntaxAccuracy"] = round(sum(syn) / len(syn), 4) if syn else None

    # AST similarity (avg cosine over node-type bags vs reference).
    sims = []
    for p, r in zip(predictions, references):
        a, b = _ast_node_types(p), _ast_node_types(r)
        if a is not None and b is not None:
            sims.append(_bag_cosine(a, b))
    report["AST_Similarity"] = round(
        sum(sims) / len(sims), 4) if sims else None

    # Compilation rate.
    rate, method = compilation_rate(predictions)
    report["CompilationRate"] = {"value": (round(rate, 4) if rate is not None else None),
                                 "method": method}
    return report

In [57]:
# Generate Stage 2 predictions on a validation subset and produce the report.
EVAL_N = min(CFG.STAGE2_EVAL_SAMPLES, len(val_pairs))
eval_pairs = val_pairs[:EVAL_N]

cs_predictions: List[str] = []
cs_references: List[str] = []
for i, p in enumerate(eval_pairs):
    try:
        pred = generate_csharp_from_java(p["java"], max_new_tokens=400)
    except Exception:
        logger.exception("Stage 2 eval generation failed at index %d.", i)
        pred = ""
    cs_predictions.append(pred)
    cs_references.append(p["cs"])
    if (i + 1) % 10 == 0:
        logger.info("  evaluated %d/%d", i + 1, EVAL_N)

stage2_report = evaluate_csharp(cs_predictions, cs_references)

logger.info("Stage 2 evaluation finished on %d samples (dataset=%s).",
            EVAL_N, DATASET_NAME)
print("\n===== Stage 2 (Java -> C#) Evaluation Report =====")
print(f"Dataset: {DATASET_NAME} | Eval samples: {EVAL_N}")
print(json.dumps(stage2_report, indent=2))

06:37:06 | INFO | QwenCodeGen |   evaluated 10/50
06:37:42 | INFO | QwenCodeGen |   evaluated 20/50
06:38:08 | INFO | QwenCodeGen |   evaluated 30/50
06:38:35 | INFO | QwenCodeGen |   evaluated 40/50
06:39:09 | INFO | QwenCodeGen |   evaluated 50/50
06:39:10 | INFO | QwenCodeGen | Stage 2 evaluation finished on 50 samples (dataset=CodeXGLUE-Java-CS).



===== Stage 2 (Java -> C#) Evaluation Report =====
Dataset: CodeXGLUE-Java-CS | Eval samples: 50
{
  "BLEU": 81.6333,
  "CodeBLEU": {
    "codebleu": 0.7546,
    "ngram_match_score": 0.7094,
    "weighted_ngram_match_score": 0.7199,
    "syntax_match_score": 0.7749,
    "dataflow_match_score": 0.8141
  },
  "ExactMatch": 0.38,
  "SyntaxAccuracy": 0.92,
  "AST_Similarity": 0.9836,
  "CompilationRate": {
    "value": 0.92,
    "method": "syntax-proxy(tree-sitter)"
  }
}


## 20. Save adapters, tokenizer & training configuration

Saves both stage adapters, the tokenizer, and a JSON of the training/eval setup into a
**timestamped** run folder (`CFG.SAVE_DIR/run_<timestamp>/`) so successive runs never overwrite
each other. A merged FP16 export of the Stage 2 model is included but commented out (optional — large).

In [59]:
# ---------------------------------------------------------------------------
# Persist everything needed to reload both stages later (timestamped run folder).
# ---------------------------------------------------------------------------
def save_all_artifacts() -> str:
    """Save both adapters, tokenizers, config + metrics. Returns the run directory."""
    run_dir = os.path.join(CFG.SAVE_DIR, f"run_{TIMESTAMP}")
    stage1_dir = os.path.join(run_dir, "stage1_nl2java_adapter")
    stage2_dir = os.path.join(run_dir, "stage2_java2csharp_adapter")
    os.makedirs(run_dir, exist_ok=True)

    try:
        # Stage 1 adapter + tokenizer (NOT retrained — persisting the in-memory adapter).
        model.save_pretrained(stage1_dir)
        tokenizer.save_pretrained(stage1_dir)
        # Stage 2 adapter + tokenizer.
        model_s2.save_pretrained(stage2_dir)
        tokenizer_s2.save_pretrained(stage2_dir)
        logger.info("Saved Stage 1 + Stage 2 adapters and tokenizers.")
    except Exception:
        logger.exception("Failed to save model adapters / tokenizers.")
        raise

    # Training / evaluation configuration (+ both stages' metrics).
    training_configuration = {
        "timestamp": TIMESTAMP,
        "base_model": CFG.MODEL_NAME,
        "stage1": {
            "task": "NL -> Java",
            "dataset": f"{CFG.DATASET_NAME}/{CFG.DATASET_CONFIG}",
            "lora": {"r": CFG.LORA_R, "lora_alpha": CFG.LORA_ALPHA,
                     "lora_dropout": CFG.LORA_DROPOUT, "bias": CFG.LORA_BIAS},
            "max_seq_length": CFG.MAX_LENGTH,
            "epochs": CFG.EPOCHS,
            "train_samples": CFG.STAGE1_TRAIN_SAMPLES,
            "hf_repo": CFG.HF_REPO,
            "evaluation": globals().get("stage1_evaluation"),
        },
        "stage2": {
            "task": "Java -> C#",
            "dataset": DATASET_NAME,
            "lora": {"r": CFG.LORA_R, "lora_alpha": CFG.STAGE2_LORA_ALPHA,
                     "lora_dropout": CFG.LORA_DROPOUT, "bias": CFG.LORA_BIAS},
            "max_seq_length": STAGE2_MAX_SEQ_LEN,
            "optimizer": stage2_config.optim,
            "fp16": stage2_config.fp16,
            "epochs": stage2_config.num_train_epochs,
            "per_device_train_batch_size": stage2_config.per_device_train_batch_size,
            "gradient_accumulation_steps": stage2_config.gradient_accumulation_steps,
            "learning_rate": stage2_config.learning_rate,
            "train_samples": len(train_pairs),
            "val_samples": len(val_pairs),
            "hf_repo": CFG.HF_REPO_STAGE2,
            "evaluation": stage2_report,
        },
    }

    config_path = os.path.join(run_dir, "training_configuration.json")
    with open(config_path, "w", encoding="utf-8") as fh:
        json.dump(training_configuration, fh, indent=2, default=str)

    logger.info("Saved configuration + metrics to %s", config_path)
    logger.info("Artifacts written to:\n - %s\n - %s\n - %s",
                stage1_dir, stage2_dir, config_path)
    return run_dir


SAVE_RUN_DIR = save_all_artifacts()

# Optional: merged FP16 export of the Stage 2 model (large; uncomment to use).
# model_s2.save_pretrained_merged(
#     os.path.join(SAVE_RUN_DIR, "stage2_java2csharp_merged"),
#     tokenizer_s2,
#     save_method="merged_16bit",
# )

06:54:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06:54:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
06:54:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06:54:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/992519aff1d7e99c87d9ebfa5eddb302e66cde3f/config.json "HTTP/1.1 200 OK"
06:54:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06:54:34 | INFO | httpx | HTTP Request: HEAD https://huggingface.c

In [60]:
# ---------------------------------------------------------------------------
# Merge BOTH stages to FP16 and push each to the Hugging Face Hub.
#
# The two stages are SEPARATE task adapters (Stage 1: NL->Java, Stage 2: Java->C#)
# applied *sequentially* in the pipeline, so they cannot be fused into a single set
# of weights. Instead each LoRA adapter is merged into its own FP16 base copy and
# pushed to its own repo:
#   - Stage 1 merged -> CFG.HF_REPO
#   - Stage 2 merged -> CFG.HF_REPO_STAGE2
# ---------------------------------------------------------------------------
PUSH_BOTH_STAGES_TO_HUB = True

# Reuse the token helper / value from Section 12 if present, else fall back to env vars.
_hf_token = globals().get("secret_value_0")
if not _hf_token:
    if "get_hf_token" in globals():
        _hf_token = get_hf_token()
    else:
        _hf_token = os.environ.get(
            "HUGGINGFACE_TOKEN") or os.environ.get("HF_TOKEN")

if PUSH_BOTH_STAGES_TO_HUB and _hf_token:
    # --- Stage 1: NL -> Java (merged FP16) ---
    try:
        logger.info("Pushing merged Stage 1 model to '%s' ...", CFG.HF_REPO)
        model.push_to_hub_merged(
            CFG.HF_REPO, tokenizer, save_method="merged_16bit", token=_hf_token)
        logger.info("Stage 1 push complete: %s", CFG.HF_REPO)
    except Exception:
        logger.exception("Failed to push Stage 1 merged model to the Hub.")
        raise

    # --- Stage 2: Java -> C# (merged FP16) ---
    try:
        logger.info("Pushing merged Stage 2 model to '%s' ...",
                    CFG.HF_REPO_STAGE2)
        model_s2.push_to_hub_merged(
            CFG.HF_REPO_STAGE2, tokenizer_s2, save_method="merged_16bit", token=_hf_token)
        logger.info("Stage 2 push complete: %s", CFG.HF_REPO_STAGE2)
    except Exception:
        logger.exception("Failed to push Stage 2 merged model to the Hub.")
        raise

    logger.info("Both stages merged and pushed:\n - %s\n - %s",
                CFG.HF_REPO, CFG.HF_REPO_STAGE2)
else:
    logger.warning("Skipping Hub push (PUSH_BOTH_STAGES_TO_HUB=%s, token_present=%s).",
                   PUSH_BOTH_STAGES_TO_HUB, bool(_hf_token))

06:54:42 | INFO | QwenCodeGen | Pushing merged Stage 1 model to 'shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_v2' ...
06:54:42 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06:54:42 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06:54:42 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-Coder-1.5B-Instruct/b9f2b422413d50e652bf90364af0efe50b48f92e/config.json "HTTP/1.1 200 OK"
06:54:42 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/qwen2.5-coder-1.5b-instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
06:54:42 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-Coder-1.5B-Instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
06:54:

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
06:54:43 | WARNING | huggingface_hub.hf_api | No files have been modified since last commit. Skipping to prevent empty commit.
06:54:44 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_v2/revision/main "HTTP/1.1 200 OK"


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

06:56:15 | INFO | httpx | HTTP Request: POST https://huggingface.co/api/models/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2/commit/main "HTTP/1.1 200 OK"
Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:44<00:00, 44.38s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2`


06:56:17 | INFO | QwenCodeGen | Stage 2 push complete: shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2
06:56:17 | INFO | QwenCodeGen | Both stages merged and pushed:
 - shibsankardhara2/Qwen2.5-Coder-1.5B-NL-Java_v2
 - shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2


## 21. Summary

This notebook fine-tunes `Qwen2.5-Coder-1.5B` into a two-stage NL → Java → C# code generator.

**Reproduce a full run (Kaggle / Colab T4):** run the cells top-to-bottom. All knobs live in the
`CFG` object (Section 3); nothing else needs editing.

**Artifacts produced** (Section 20, in `CFG.SAVE_DIR/run_<timestamp>/`):
- `stage1_nl2java_adapter/` — Stage 1 LoRA adapter + tokenizer
- `stage2_java2csharp_adapter/` — Stage 2 LoRA adapter + tokenizer
- `training_configuration.json` — hyperparameters + Stage 1 & Stage 2 evaluation metrics

**Key public functions**
| Function | Purpose |
|----------|---------|
| `generate_java_from_nl(prompt)` | NL → Java (Stage 1) |
| `generate_csharp_from_java(java)` | Java → C# (Stage 2) |
| `generate_csharp_from_nl(prompt)` | Chained NL → Java → C# |
| `compute_code_metrics(refs, preds)` | CodeBLEU + CodeBERTScore |
| `evaluate_csharp(preds, refs)` | BLEU / CodeBLEU / ExactMatch / Syntax / AST / Compilation |

**Refactor highlights:** centralized `CFG`, structured logging, exception handling on every stage,
seeded RNGs, batched preprocessing, de-duplicated inference/eval helpers, validation guards, and
timestamped artifact saving. Heavy/optional dependencies are imported lazily so a single missing
metric package never blocks the run.